# Detailed Notebook 1: Solving Equations, Fitting, and Minimizing

A core skill in computational physics is finding where equations are satisfied
and extracting parameters that best describe data.  This notebook walks through
every major technique with **step-by-step explanations** and multiple worked
examples drawn from the class notes.

**Contents**
1. Scalar root finding with `root_scalar`
2. Solving coupled nonlinear systems with `root`
3. Fitting a model to data with `curve_fit`
4. General minimization with `minimize`
5. Assessing fit quality with chi-squared

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar, root, curve_fit, minimize
from scipy.stats import chi2
from scipy.integrate import quad

---
## Section 1: Scalar root finding with `root_scalar`

### The idea

A *root* of a function f(x) is any value x* where **f(x*) = 0**.
In physics we constantly rewrite equations into this form:

* "At what angle does the pendulum period equal 2 s?" → T(θ) − 2 = 0
* "Where does the projectile hit the ground?" → y(t) = 0
* "What frequency gives maximum power?" → dP/df = 0

### How the Brent method works (what happens inside `root_scalar`)

1. You supply a *bracket* [a, b] where f(a) and f(b) have **opposite signs**.
   By the intermediate value theorem, at least one root lies between them.
2. `brentq` iteratively **narrows the bracket** (like binary search),
   using a clever mix of bisection and inverse quadratic interpolation.
3. It stops when the bracket width is less than machine precision (~1e-15).

### Function signature

```python
sol = root_scalar(f, bracket=[a, b], method='brentq')
sol.root       # the x-value that makes f(x)=0
sol.converged  # True if it succeeded
```

In [ ]:
# ── Example 1a: solve  cos(x) = x  ─────────────────────────────────────────
# Step 1: rearrange to f(x) = 0.
#   cos(x) = x   →   cos(x) - x = 0
# Step 2: check that there is a sign change somewhere (so we can bracket).
#   f(0) = cos(0) - 0 = 1 > 0
#   f(1) = cos(1) - 1 ≈ -0.46 < 0
# Step 3: call root_scalar with that bracket.

f = lambda x: np.cos(x) - x

print('f(0) =', round(f(0), 6))   # should be > 0
print('f(1) =', round(f(1), 6))   # should be < 0

sol = root_scalar(f, bracket=[0, 1], method='brentq')
print('root x* =', sol.root)
print('converged:', sol.converged)
print('verification f(x*) =', f(sol.root), '  (should be ~0)')

**What just happened?**

- We turned the equation `cos(x) = x` into a root problem `f(x) = cos(x) - x = 0`.
- The sign check confirms there is exactly one root in [0, 1].
- `brentq` zeroes in on it to ~15 significant figures in only ~15 iterations.
- The residual `f(sol.root)` should be essentially zero (< 1e-14).

In [ ]:
# ── Visualise the root ───────────────────────────────────────────────────────
x_plot = np.linspace(-0.5, 2, 400)
plt.figure(figsize=(7, 4))
plt.plot(x_plot, f(x_plot), label='f(x) = cos(x) − x')
plt.axhline(0, color='k', linewidth=0.8)
plt.axvline(sol.root, color='red', linestyle='--',
            label=f'root = {sol.root:.5f}')
plt.scatter([sol.root], [0], color='red', zorder=5, s=60)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Fixed point of cos(x) = x')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Reading the plot:**
The red dashed line marks x* ≈ 0.7391.  The curve f(x) = cos(x) − x crosses
zero exactly there, meaning cos(x*) = x*.  This is called a *fixed point*.

In [ ]:
# ── Example 1b: at what amplitude does the pendulum period equal 2.5 s? ─────
#
# Physics: the *exact* period of a simple pendulum of length L is
#
#   T(A) = 4 * sqrt(L/g) * K(sin(A/2))
#
# where K is the complete elliptic integral of the first kind.
# For small A, T → 2π√(L/g) (the familiar formula).
# For larger A, T increases.  We want to find A such that T(A) = 2.5 s.
#
# Method: define g(A) = T(A) - T_target = 0 and use root_scalar.

L = 1.0          # pendulum length in metres
g_acc = 9.81     # gravitational acceleration
T_target = 2.5   # desired period in seconds

def pendulum_period(A):
    # Full elliptic-integral formula; A is amplitude in radians
    integrand = lambda phi: 1 / np.sqrt(1 - np.sin(A/2)**2 * np.sin(phi)**2)
    K, _ = quad(integrand, 0, np.pi/2)
    return 4 * np.sqrt(L / g_acc) * K

# Small-angle period for reference
T_small = 2 * np.pi * np.sqrt(L / g_acc)
print(f'Small-angle period T0 = {T_small:.4f} s  (T_target = {T_target} s requires larger angle)')

# Define root equation: g(A) = 0
g_pend = lambda A: pendulum_period(A) - T_target

# f at A≈0 gives T≈T0 < 2.5, so g < 0 there; at A→π, T→∞ so g > 0
# Bracket from 0.01 to 3 rad
sol_pend = root_scalar(g_pend, bracket=[0.01, 3.0], method='brentq')
A_star = sol_pend.root
print(f'Required amplitude A* = {np.degrees(A_star):.2f} degrees')
print(f'Verification: T(A*) = {pendulum_period(A_star):.4f} s')

**Step-by-step walkthrough:**

1. `pendulum_period(A)` computes the *exact* period using numerical integration (see Notebook 3).
2. We subtract the target T = 2.5 s to make a root problem: g(A) = T(A) − 2.5 = 0.
3. We check signs: at small A the period is 2.006 s < 2.5 (g < 0);
   at A → π rad the period → ∞ > 2.5 (g > 0).  So there is a root in between.
4. `brentq` finds it: about **65 degrees** of amplitude is needed.

**Physical insight:** the small-angle approximation underestimates how large the
amplitude must be if you need a slow swing.

---
## Section 2: Solving coupled nonlinear systems with `root`

### The idea

When you have **N unknowns** and **N equations**, you need to satisfy all of them
simultaneously.  `root` takes a *vector function* F(v) that should return an array
of N residuals.  It drives all of them to zero at once.

### How it works internally

`root` uses Newton's method by default:
- Start at initial guess x0.
- Estimate the Jacobian (how each equation changes with each variable).
- Take a Newton step toward the zero.
- Repeat until the residual norm is below tolerance.

### Function signature

```python
sol = root(F, x0=[guess_1, guess_2, ...])
sol.x        # solution vector
sol.success  # True/False
sol.fun      # residuals at the solution (should all be ~0)
```

**Tip:** the initial guess `x0` matters — if there are multiple solutions, different
guesses will lead to different ones.  Pick something physically motivated.

In [ ]:
# ── Example 2a: two coupled equations ───────────────────────────────────────
# Find (x, y) satisfying:
#   (1)  x^2 + y^2 = 5    (point lies on a circle of radius sqrt(5))
#   (2)  x - y = 1        (point lies on a diagonal line)
#
# Rearranged to residuals = 0:
#   r1(x,y) = x^2 + y^2 - 5 = 0
#   r2(x,y) = x - y - 1    = 0

def F_circle_line(v):
    x, y = v[0], v[1]
    return [x**2 + y**2 - 5,   # residual of equation 1
            x - y - 1]          # residual of equation 2

sol2a = root(F_circle_line, x0=[2.0, 1.0])

print('solution [x, y] =', sol2a.x)
print('residuals at solution =', F_circle_line(sol2a.x))   # should be ~[0, 0]
print('success:', sol2a.success)

# Manual sanity check
x_s, y_s = sol2a.x
print(f'\nCheck:  x^2 + y^2 = {x_s**2 + y_s**2:.8f}  (want 5.0)')
print(f'Check:  x - y     = {x_s - y_s:.8f}  (want 1.0)')

**What just happened?**

- `F_circle_line(v)` takes a vector `v = [x, y]` and returns **two residuals** —
  one per equation.  When both residuals are zero, the system is solved.
- We started the solver at (2, 1), which is near the actual answer (2.0, 1.0 exactly would be exact here since x=2, y=1 satisfies both: 4+1=5 ✓, 2-1=1 ✓).
- `sol2a.fun` (same as `F_circle_line(sol2a.x)`) shows residuals near machine
  precision, confirming convergence.

In [ ]:
# ── Example 2b: two parabolas — finding both intersection points ─────────────
# Curves:
#   y = x^2 - 1    (upward parabola)
#   y = -x^2 + 3   (downward parabola)
# Setting equal: x^2 - 1 = -x^2 + 3 → x = ±√2
# (Two intersections — we'll find each by using different initial guesses.)

def F_parabolas(v):
    x, y = v[0], v[1]
    return [x**2 - 1 - y,       # y = x^2 - 1  rearranged
            -x**2 + 3 - y]      # y = -x^2 + 3 rearranged

# Right intersection: x ≈ +√2 ≈ 1.41
sol_right = root(F_parabolas, x0=[1.5, 1.0])
# Left intersection: x ≈ -√2 ≈ -1.41
sol_left  = root(F_parabolas, x0=[-1.5, 1.0])

print('Right intersection (x, y) =', np.round(sol_right.x, 6))
print('Left  intersection (x, y) =', np.round(sol_left.x, 6))

x_plot = np.linspace(-2.5, 2.5, 300)
plt.figure(figsize=(6, 4))
plt.plot(x_plot, x_plot**2 - 1, label='y = x² − 1')
plt.plot(x_plot, -x_plot**2 + 3, label='y = −x² + 3')
plt.scatter([sol_right.x[0], sol_left.x[0]],
            [sol_right.x[1], sol_left.x[1]],
            color='red', zorder=5, s=80, label='roots found by `root`')
plt.legend()
plt.xlabel('x')
plt.ylabel('y')
plt.title('Two intersections found with different initial guesses')
plt.grid(True, alpha=0.3)
plt.show()

**Key lesson:** When multiple solutions exist (multiple intersection points, multiple
equilibria, etc.), run `root` several times with different initial guesses — each
will converge to a different solution.

In [ ]:
# ── Example 2c: LC circuit resonance ────────────────────────────────────────
# Impedance of a series RLC circuit:
#   Z(ω) = R + j(ωL - 1/(ωC))
# At resonance, Im(Z) = 0:
#   ωL - 1/(ωC) = 0   →   ω = 1/√(LC)
# Let's find it numerically just to practice `root`.

R, L_circ, C_circ = 10.0, 1e-3, 1e-6   # ohms, henrys, farads

def imag_impedance(omega_arr):
    omega = omega_arr[0]
    imZ = omega * L_circ - 1.0 / (omega * C_circ)
    return [imZ]

omega_analytic = 1.0 / np.sqrt(L_circ * C_circ)
print(f'Analytic resonance: ω0 = {omega_analytic:.2f} rad/s')

sol_LC = root(imag_impedance, x0=[30000.0])
print(f'Numerical resonance: ω0 = {sol_LC.x[0]:.2f} rad/s')

---
## Section 3: Fitting a model to data with `curve_fit`

### The idea

In experiments we have noisy measurements and a theoretical model with unknown
parameters.  `curve_fit` finds the parameter values that make the model best
match the data by **minimising the sum of squared residuals**:

    SSR = Σ_i  [y_data[i] − model(x[i], *params)]²

This is called *least-squares fitting*.

### Function signature

```python
popt, pcov = curve_fit(model, x_data, y_data, p0=[initial_guess])
```

- `popt` — optimal (best-fit) parameter values
- `pcov` — covariance matrix; `np.sqrt(np.diag(pcov))` gives 1σ uncertainties
- `p0` — initial parameter guess (required for nonlinear models!)

### Model function convention

```python
def model(x, param1, param2, ...):
    return ...
```

The first argument is **always** the independent variable x.
The remaining arguments are the parameters to be fitted.

In [ ]:
# ── Example 3a: linear fit  y = a*x + b ─────────────────────────────────────
rng = np.random.default_rng(42)
x_data = np.linspace(0, 5, 30)
true_a, true_b = 2.4, -0.7
# Simulate noisy data: true line + Gaussian noise with sigma=0.5
y_data = true_a * x_data + true_b + rng.normal(0, 0.5, size=x_data.size)

def linear_model(x, a, b):
    return a * x + b

# p0 is the initial guess — for a linear model it doesn't matter much.
popt, pcov = curve_fit(linear_model, x_data, y_data, p0=[1.0, 0.0])
perr = np.sqrt(np.diag(pcov))   # 1-sigma uncertainties from covariance matrix

print('curve_fit result:')
print(f'  a = {popt[0]:.4f} ± {perr[0]:.4f}   (true: {true_a})')
print(f'  b = {popt[1]:.4f} ± {perr[1]:.4f}   (true: {true_b})')

**What each quantity means:**

| Variable | Meaning |
|----------|---------|
| `popt[0]` | Best-fit slope `a` |
| `popt[1]` | Best-fit intercept `b` |
| `perr[0]` | 1σ uncertainty on `a`; the true `a` should fall within `popt[0] ± perr[0]` ~68% of the time |
| `perr[1]` | 1σ uncertainty on `b` |

`pcov` (the covariance matrix) is computed from the curvature of the SSR surface at the minimum.
A narrower SSR valley (more data, less noise) → smaller `pcov` → tighter parameter uncertainty.

In [ ]:
# Visualise the fit
x_fine = np.linspace(0, 5, 300)
plt.figure(figsize=(7, 4))
plt.scatter(x_data, y_data, s=25, label='noisy data', zorder=5)
plt.plot(x_fine, linear_model(x_fine, *popt), 'r',
         label=f'fit: {popt[0]:.2f}x + {popt[1]:.2f}')
plt.plot(x_fine, linear_model(x_fine, true_a, true_b), 'k--',
         label='truth', linewidth=1.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('curve_fit: linear model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── Example 3b: exponential decay  y = A*exp(-k*t) + C ──────────────────────
# Physics context: radioactive source — signal decays over time.
rng2 = np.random.default_rng(7)
t_data = np.linspace(0, 5, 40)
A_true, k_true, C_true = 3.0, 0.9, 0.15
y_decay = A_true * np.exp(-k_true * t_data) + C_true \
          + rng2.normal(0, 0.05, t_data.size)

def exp_model(t, A, k, C):
    # A: initial amplitude, k: decay rate, C: baseline/background
    return A * np.exp(-k * t) + C

# Good initial guess: A ≈ first data value, k ≈ 1, C ≈ 0
p0_exp = [y_decay[0], 1.0, 0.0]
popt2, pcov2 = curve_fit(exp_model, t_data, y_decay, p0=p0_exp)
perr2 = np.sqrt(np.diag(pcov2))

print('Exponential decay fit:')
print(f'  A = {popt2[0]:.4f} ± {perr2[0]:.4f}   (true: {A_true})')
print(f'  k = {popt2[1]:.4f} ± {perr2[1]:.4f}   (true: {k_true})')
print(f'  C = {popt2[2]:.4f} ± {perr2[2]:.4f}   (true: {C_true})')

t_fine = np.linspace(0, 5, 300)
plt.figure(figsize=(7, 4))
plt.scatter(t_data, y_decay, s=25, label='data')
plt.plot(t_fine, exp_model(t_fine, *popt2), 'r', label='fit')
plt.plot(t_fine, exp_model(t_fine, A_true, k_true, C_true), 'k--', label='truth')
plt.xlabel('t (s)')
plt.ylabel('signal')
plt.title('curve_fit: exponential decay')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Why does `p0` matter here?**

For nonlinear models like an exponential, `curve_fit` uses an iterative algorithm.
If the starting point `p0` is far from the true answer, the algorithm can:
- Converge to a wrong local minimum
- Fail to converge entirely

A physically motivated guess (amplitude ≈ first data value, decay rate ≈ 1)
helps it find the correct global minimum.

In [ ]:
# ── Example 3c: Gaussian peak  (common in spectroscopy and detector work) ────
rng3 = np.random.default_rng(13)
E_data = np.linspace(0, 10, 60)
peak_E, peak_sig, peak_amp, bg = 5.0, 0.8, 4.0, 0.3
y_spec = peak_amp * np.exp(-0.5*((E_data - peak_E)/peak_sig)**2) + bg \
         + rng3.normal(0, 0.1, E_data.size)

def gaussian_model(E, amp, mu, sigma, bg):
    # amp: peak height, mu: centre energy, sigma: width, bg: flat background
    return amp * np.exp(-0.5 * ((E - mu) / sigma)**2) + bg

# Smart initial guess: look at the data to estimate amp, centre, width
p0_g = [y_spec.max() - y_spec.min(),    # amplitude ≈ max minus baseline
        E_data[np.argmax(y_spec)],       # centre ≈ x at the max
        1.0,                             # width ≈ rough estimate
        y_spec.min()]                    # baseline ≈ min value
popt3, pcov3 = curve_fit(gaussian_model, E_data, y_spec, p0=p0_g)
perr3 = np.sqrt(np.diag(pcov3))

print('Gaussian peak fit:')
print(f'  amplitude = {popt3[0]:.4f} ± {perr3[0]:.4f}  (true: {peak_amp})')
print(f'  centre    = {popt3[1]:.4f} ± {perr3[1]:.4f}  (true: {peak_E})')
print(f'  sigma     = {popt3[2]:.4f} ± {perr3[2]:.4f}  (true: {peak_sig})')
print(f'  baseline  = {popt3[3]:.4f} ± {perr3[3]:.4f}  (true: {bg})')

E_fine = np.linspace(0, 10, 300)
plt.figure(figsize=(7, 4))
plt.scatter(E_data, y_spec, s=20, label='data')
plt.plot(E_fine, gaussian_model(E_fine, *popt3), 'r', label='Gaussian fit')
plt.xlabel('Energy (arb)')
plt.ylabel('Counts/bin')
plt.title('curve_fit: Gaussian peak')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Section 4: General minimization with `minimize`

### When to use `minimize` instead of `curve_fit`

`curve_fit` is specialised for least-squares fitting and returns uncertainties.
Use `minimize` when:

- You have a **custom objective** (e.g. Poisson log-likelihood, not SSR)
- You need **bounds** (parameters must be positive, or within a range)
- You need **constraints** (one parameter must equal a function of another)
- Your model is complex enough that you prefer full control

### The objective function

The key concept is the *objective function* f(params) → scalar.
You want to **minimise** this.  Common choices:

| Objective | When to use |
|-----------|-------------|
| Sum of squared residuals (SSR) | Gaussian noise, ordinary least squares |
| Weighted SSR: Σ (y-ŷ)²/σ² | Data points have known uncertainties σ |
| Negative log-likelihood | Poisson counts, maximum likelihood estimation |

### Function signature

```python
from scipy.optimize import minimize, Bounds

result = minimize(objective, x0=[start], bounds=Bounds(lb=[lo], ub=[hi]))
result.x    # best-fit parameters
result.fun  # minimum objective value
result.success
```

In [ ]:
# ── Example 4a: fit a parabola y = a*x^2 + b*x + c ─────────────────────────
rng4 = np.random.default_rng(99)
x_par = np.linspace(-2, 2, 40)
y_par = 1.5*x_par**2 - 0.3*x_par + 0.8 + rng4.normal(0, 0.2, x_par.size)

def sse_parabola(p):
    # p is a length-3 array [a, b, c]
    a, b, c = p
    y_model = a*x_par**2 + b*x_par + c
    # SSE = sum of (data - model)^2
    return np.sum((y_par - y_model)**2)

# Start from a neutral guess — minimizer will explore and find the valley bottom
result4a = minimize(sse_parabola, x0=[1.0, 0.0, 0.0])

print('Parabola fit via minimize:')
print(f'  a = {result4a.x[0]:.4f}  (true: 1.5)')
print(f'  b = {result4a.x[1]:.4f}  (true: -0.3)')
print(f'  c = {result4a.x[2]:.4f}  (true: 0.8)')
print(f'  minimum SSE = {result4a.fun:.6f}')
print(f'  success: {result4a.success}')

**Step-by-step walkthrough of `minimize`:**

1. `minimize` evaluates `sse_parabola([1, 0, 0])` at the starting point.
2. It perturbs each parameter slightly to estimate how the SSE changes in each direction (the gradient).
3. It takes a step in the direction that decreases SSE most.
4. It repeats until the SSE stops decreasing (gradient ≈ 0).
5. The algorithm used by default (L-BFGS-B) is efficient for smooth objectives.

In [ ]:
# ── Example 4b: bounded fit — decay with non-negative parameters ─────────────
# Physics: count rate N(t) = N0 * exp(-t/tau)
# N0 > 0 (initial count), tau > 0 (lifetime).  We enforce this with bounds.

from scipy.optimize import Bounds

rng5 = np.random.default_rng(33)
t_counts = np.linspace(0, 10, 25)
N0_true, tau_true = 500.0, 3.5
y_counts = N0_true * np.exp(-t_counts/tau_true) + rng5.normal(0, 10, t_counts.size)
y_counts = np.maximum(y_counts, 0)   # counts can't be negative

def sse_decay(p):
    N0, tau = p
    y_model = N0 * np.exp(-t_counts / tau)
    return np.sum((y_counts - y_model)**2)

# Bounds: N0 in [0, 10000], tau in [0.1, 100]
bounds = Bounds(lb=[0, 0.1], ub=[10000, 100])
result4b = minimize(sse_decay, x0=[400.0, 2.0], bounds=bounds)

print(f'N0  = {result4b.x[0]:.2f}  (true: {N0_true})')
print(f'tau = {result4b.x[1]:.4f} s  (true: {tau_true})')

t_fine = np.linspace(0, 10, 300)
plt.figure(figsize=(7, 4))
plt.scatter(t_counts, y_counts, s=25, label='data')
plt.plot(t_fine, result4b.x[0]*np.exp(-t_fine/result4b.x[1]), 'r', label='fit')
plt.xlabel('t (s)')
plt.ylabel('counts')
plt.title('minimize with bounds: decay fit')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Section 5: Assessing fit quality with chi-squared

### The idea

After fitting you must ask: **is this fit actually good?**
Residuals close to zero are necessary but not sufficient — a model with enough
free parameters can always fit anything.

The **reduced chi-squared** provides a statistical test:

    χ²_red = SSE / (σ² × dof)

where:
- `SSE` = sum of squared residuals at the best fit
- `σ` = measurement uncertainty on each data point
- `dof` = degrees of freedom = N_data − N_parameters

### Interpreting χ²_red

| χ²_red | Interpretation |
|--------|----------------|
| ≈ 1 | Model fits the data — scatter is consistent with σ |
| >> 1 | Model is wrong, or σ is underestimated |
| << 1 | Model overfits, or σ is overestimated |

In [ ]:
# Using the linear fit from Example 3a
sigma = 0.5   # we know the true noise level
residuals = y_data - linear_model(x_data, *popt)
SSE = np.sum(residuals**2)

n_data   = len(y_data)
n_params = 2          # fitted a and b
dof      = n_data - n_params

chi2_red = SSE / (sigma**2 * dof)
print(f'Number of data points:  {n_data}')
print(f'Number of parameters:   {n_params}')
print(f'Degrees of freedom:     {dof}')
print(f'SSE = {SSE:.4f}')
print(f'Reduced chi-squared χ²_red = {chi2_red:.4f}')

if 0.5 < chi2_red < 2.0:
    print('→ Good fit (χ²_red ≈ 1 as expected)')
elif chi2_red > 2.0:
    print('→ Poor fit or σ is underestimated')
else:
    print('→ Possible overfitting or σ is overestimated')

# p-value: probability of getting chi2 >= observed value by chance if model is correct
p_value = 1 - chi2.cdf(chi2_red * dof, df=dof)
print(f'\np-value = {p_value:.4f}')
print('(p < 0.05 would mean we can reject the fit at 95% confidence)')

**How to read the p-value:**

- The p-value is the probability that a *correct* model would produce chi-squared
  this large or larger just by random chance.
- If p > 0.05 we cannot reject the model — the fit is consistent with the data.
- If p < 0.05 the model is probably wrong, or the assumed σ is incorrect.

---
## Summary and decision guide

```
Do I need to find where an equation equals zero?
  → One variable   →  root_scalar   (fast, reliable with a bracket)
  → Many variables →  root          (Newton-type; provide a good initial guess)

Do I need to fit a model to data?
  → Model is known analytically →  curve_fit   (returns uncertainties too)
  → Custom objective / bounds   →  minimize

After fitting, always:
  1. Plot the fit against the data
  2. Plot the residuals (data − model) — they should scatter randomly around zero
  3. Compute chi-squared to check statistical quality
```